# 桶排序
对MA因子进行传统的桶排序，并且构建对冲组合，分析每个桶的收益和夏普。 

不同的任务都可以使用该脚本  

异质性分析数据需要预先筛选因子数据  




## 导入库

In [1]:
import warnings
from typing import Any
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as t_dist

import os
import dotenv
dotenv.load_dotenv()

True

## 超参数 

In [2]:
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'baseline2'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

BUCKETS_NUM = 10 # 分桶数  
HETER = False # 如果不使用异质性，则导入MA_copy，保证不存在异质性

RISK_FREE_RATE = 0 # 无风险利率  

## 读取数据  
需要读取AED数据和市值数据  
市值数据：数据库 or 本地json (测试用)   


In [3]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子.parquet') if HETER else pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2017-08-01,"""601011""",0.78383,0.180845
2018-01-01,"""300398""",0.68997,0.0365
2018-08-01,"""300079""",0.767654,0.0198
2022-09-01,"""000823""",0.6550625,0.082313
2022-07-01,"""300654""",0.794424,0.0468


In [4]:
# 市值数据从 DB 读取（statics.market_value）
market_value_df = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='''SELECT stkcd AS "Stkcd", trdmnt AS "Trdmnt", msmvosd AS "Msmvosd" FROM statics.market_value''',
    engine=ENGINE,
)


## 处理数据(测试)

In [5]:
market_value_df = market_value_df.rename(
    {
        'Stkcd':'portfolio',
        'Trdmnt':'date',
        'Msmvosd':'msmvosd'
    }
)

market_value_df.head()

portfolio,date,msmvosd
str,date,f64
"""000557""",1997-05-01,963245.03
"""000031""",1997-07-01,1.3565e6
"""600835""",1997-09-01,177336.0
"""900920""",1997-03-01,107198.0
"""000417""",1997-06-01,415800.0


## 桶排序  
按照AED因子对数据进行桶排序，按分位数构建BUCKETS_NUM个组合，组合内部使用市值加权平均计算组合收益，然后绘制各个组合的累计收益。   
同时，还需要构建一个对冲组合，其收益为最后一个组合和第一个组合的差值。     

### 构建桶  
#### 分桶组合
首先进行分桶，获取每个桶的时序收益(series_list)


In [6]:
# 分桶
ma_df = ma_df.with_columns(
    [
        pl.col('MA').quantile(i / BUCKETS_NUM, interpolation='lower').alias(f'q{i}')
        for i in range(1,BUCKETS_NUM)
    ]
)

joined_df = ma_df.join(market_value_df, on=['portfolio','date'], how='left')
joined_df = joined_df.with_columns(
    (pl.col('msmvosd') / pl.col('msmvosd').sum().over('date')).alias('weight')
)

series_list = list[pl.DataFrame]() # 每一个桶对应的series，date-weighted_sum_ret
mean_ret_list = []  # 每一个桶的月平均收益  
for i in range(0,BUCKETS_NUM): # 按照MA由小到大排序  
    # 第1个组合
    if i == 0:
        bucket_df = joined_df.filter(pl.col(f'MA') <= pl.col(f'q1'))
    elif i == (BUCKETS_NUM - 1):
        bucket_df = joined_df.filter(pl.col(f'MA') > pl.col(f'q{i}'))
    else:
        bucket_df = joined_df.filter((pl.col(f'MA') > pl.col(f'q{i}')) & (pl.col(f'MA') <= pl.col(f'q{i+1}')))
    bucket_df = bucket_df.select(['date','portfolio','weight','return'])
    
    # 获取series
    series_df = bucket_df.group_by(['date']).agg(
        (pl.col('weight') * pl.col('return')).sum().alias('weighted_sum_ret'),
        pl.lit(i).alias('bucket_id')
    )

    # 添加到series_list
    series_list.append(series_df)

In [7]:
# 合并
combined_series = pl.concat(series_list, how='vertical').sort(['date','bucket_id'])
combined_series = combined_series.with_columns(pl.col('bucket_id').cast(pl.Utf8)) # 转为字符串，便于添加对冲组合

#### 对冲组合  
做多第BUCKET_NUM-1个组合，做空第1个组合。 

In [8]:
# 构建对冲组合
low_bucket = combined_series.filter(pl.col('bucket_id') == str(0)).rename({'weighted_sum_ret':'low_bucket_ret'})
high_bucket = combined_series.filter(pl.col('bucket_id') == str(BUCKETS_NUM - 1)).rename({'weighted_sum_ret':'high_bucket_ret'})  
joined_bucket = low_bucket.join(high_bucket, on='date', how='left')
joined_bucket = joined_bucket.with_columns(
    pl.col('high_bucket_ret').fill_null(0).alias('high_bucket_ret'),
    pl.col('low_bucket_ret').fill_null(0).alias('low_bucket_ret'),
)
joined_bucket = joined_bucket.with_columns(
    (pl.col('high_bucket_ret') - pl.col('low_bucket_ret')).alias('weighted_sum_ret')
)
joined_bucket = joined_bucket.select('date','weighted_sum_ret').with_columns(
    pl.lit('对冲组合').alias('bucket_id')
)

joined_bucket.head()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,0.002286,"""对冲组合"""
2005-01-01,-0.002428,"""对冲组合"""
2005-02-01,0.002468,"""对冲组合"""
2005-03-01,0.001375,"""对冲组合"""
2005-04-01,0.008225,"""对冲组合"""


In [9]:
combined_series = pl.concat([combined_series, joined_bucket], how='vertical').sort(['date','bucket_id'])
combined_series.head()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,-0.003841,"""0"""
2004-12-01,-0.004264,"""1"""
2004-12-01,-0.008917,"""2"""
2004-12-01,-0.006856,"""3"""
2004-12-01,-0.007445,"""4"""


In [10]:
if SAVE:
    combined_series.write_parquet(SAVE_BASE_DIR + '/分桶组合收益.parquet')

## 表现

### 样本数 
统计每个桶的样本数  

In [11]:
count = combined_series.group_by('bucket_id').agg(pl.col('weighted_sum_ret').count().alias('count'))
count.sort('bucket_id')


bucket_id,count
str,u32
"""0""",241
"""1""",241
"""2""",241
"""3""",241
"""4""",241
…,…
"""6""",241
"""7""",241
"""8""",241


### 平均收益
#### 每个桶的平均收益（包括对冲组合）    
对于每个桶序列，计算其各个时期的平均收益, HAC-t 和 p  

计算方式为，使用`weighted_sum_ret`回归常数项，使用HAC-t和p。  

In [12]:
# 每个 bucket：weighted_sum_ret 的 mean、HAC-t、p（只回归常数项，无 market_ret）
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy()
        x = np.ones((len(y), 1))
        results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f'计算{bid}时发生错误: {e}')
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [0.0],
            't': [0.0],
            'p': [1.0],
        })
mean_tp_table = combined_series.group_by('bucket_id').map_groups(regress_one)

In [13]:
mean_tp_table

bucket_id,mean_return,t,p
str,f64,f64,f64
"""对冲组合""",0.002826,2.773611,0.005544
"""5""",0.001126,1.896819,0.057852
"""1""",-0.00075,-0.743174,0.457376
"""8""",0.001149,3.349684,0.000809
"""2""",0.000207,0.231284,0.817094
…,…,…,…
"""9""",0.00125,4.594122,0.000004
"""7""",0.001189,2.769178,0.00562
"""4""",0.000659,1.018147,0.308608


In [14]:
# 格式化：4 位有效数字
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
# t、p 加括号
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    pl.col('mean_return'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
# 居中对齐到 12 位
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
# 合并为一行展示
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    (pl.col('mean_return') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('mean_return'),
).sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(mean_tp_table)

bucket_id,mean_return
str,str
"""0""",""" -0.001576 [-1.432] (0.1521) """
"""1""",""" -0.0007504 [-0.7432] (0.4574) """
"""2""",""" 0.0002074 [0.2313] (0.8171) """
"""3""",""" 0.0006079 [0.7587] (0.448) """
"""4""",""" 0.0006586 [1.018] (0.3086) """
…,…
"""6""",""" 0.001001 [2.147] (0.03175) """
"""7""",""" 0.001189 [2.769] (0.00562) """
"""8""",""" 0.001149 [3.35] (0.000809) """


将对冲组合的收益序列添加到桶排序的底部  

绘制panel_list的累计收益曲线  

绘制方法为，对于每一个时序数据，计算每一期累计收益，然后绘制成曲线。  

In [15]:
# 累加
combined_series = combined_series.with_columns(pl.col('weighted_sum_ret').cum_sum().over('bucket_id').alias('cum_return'))

# 百分化
combined_series = combined_series.with_columns(
    pl.col('cum_return').mul(100).alias('cum_return')
)

# 平滑
ROLLING_WINDOW = 4
MIN_SAMPLES = 1
combined_series = combined_series.with_columns(
    pl.col('cum_return').rolling_mean(window_size=ROLLING_WINDOW, min_samples=MIN_SAMPLES).over('bucket_id').alias('cum_return_smooth')
)

# 绘图
fig = px.line(combined_series, x='date', y='cum_return_smooth', color='bucket_id')
fig.update_layout(
    title=f'累计收益按桶分组(平滑窗口={ROLLING_WINDOW},最小样本={MIN_SAMPLES})',           # 图标题
    xaxis_title='日期',                # x 轴名称
    yaxis_title=f'累计收益率(%)',           # y 轴名称
)
fig.show()
    
    

In [16]:
if SAVE:
    #fig.write_image(SAVE_BASE_DIR + '/基准回归-累计收益率分桶图.png')
    pass 

### Sharp  
计算每个组合的Sharp比率    

计算方式为：$(mean(ret) - RISK\_FREE\_RATE) / std(ret)$


In [17]:
sharp_series = combined_series.group_by(['bucket_id']).agg(
    pl.col('weighted_sum_ret').mean().alias('mean_ret'),
    pl.col('weighted_sum_ret').std().alias('std_ret'),
)

sharp_series = sharp_series.select(
    pl.col('bucket_id'),
    ((pl.col('mean_ret') - RISK_FREE_RATE) / pl.col('std_ret')).alias('sharp')
)

sharp_series = sharp_series.with_columns(
    pl.col('sharp').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('sharp'),
)
sharp_series = sharp_series.sort('bucket_id')
sharp_series

bucket_id,sharp
str,str
"""0""","""-0.09503"""
"""1""","""-0.05433"""
"""2""","""0.01808"""
"""3""","""0.05892"""
"""4""","""0.079"""
…,…
"""6""","""0.1673"""
"""7""","""0.2212"""
"""8""","""0.2545"""


### 合并+保存 收益和sharp

In [18]:
performance = count.join(mean_tp_table, on='bucket_id', how='left').join(sharp_series, on='bucket_id', how='left').sort('bucket_id')

if SAVE:
    performance.write_parquet(SAVE_BASE_DIR + '/分桶表现.parquet')

In [19]:
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(performance)

bucket_id,count,mean_return,sharp
str,u32,str,str
"""0""",241,""" -0.001576 [-1.432] (0.1521) ""","""-0.09503"""
"""1""",241,""" -0.0007504 [-0.7432] (0.4574) ""","""-0.05433"""
"""2""",241,""" 0.0002074 [0.2313] (0.8171) ""","""0.01808"""
"""3""",241,""" 0.0006079 [0.7587] (0.448) ""","""0.05892"""
"""4""",241,""" 0.0006586 [1.018] (0.3086) ""","""0.079"""
…,…,…,…
"""6""",241,""" 0.001001 [2.147] (0.03175) ""","""0.1673"""
"""7""",241,""" 0.001189 [2.769] (0.00562) ""","""0.2212"""
"""8""",241,""" 0.001149 [3.35] (0.000809) ""","""0.2545"""
